# Feature layout check

**In one sentence:** the features file has 115 columns with no names, and this notebook proves which statistic is in each column, so silver can name them correctly.

**Answer:** the order in the Kitsune **code** is correct. The order in the Kitsune **paper** is wrong.

---

## The problem

Bronze stores the features as `_c0`, `_c1`, ... `_c114`. That is like a spreadsheet with 115 columns and **no headers**.
In silver we want real names (for example `HH_L5_pcc` instead of `_c36`), so anyone can understand the table.

The Kitsune authors describe the columns in two places, but **the two places disagree**:

| Source | Order of the blocks | We call it |
|---|---|---|
| Kitsune code on GitHub (`netStat.py`) | MI, H, **HH, HH_jit**, HpHp | **Layout A** |
| Kitsune paper (Table II) | MI, H, **HH_jit, HH**, HpHp | **Layout B** |

One of them is wrong. If we picked one without checking, we could give the wrong name to 50 columns.
So we ask **the data** which one is right.

---

## What the 115 numbers describe

Every row is one network packet. Kitsune describes each packet with statistics about the recent traffic around it.

**1. Five points of view (blocks).** Who is talking to whom:

| Block | It looks at... | Example |
|---|---|---|
| **MI** | everything this device sends (its MAC + IP address) | "how much is this camera sending?" |
| **H** | everything this IP sends | same idea, using only the IP |
| **HH** | the conversation between two IPs | "camera and server" |
| **HH_jit** | the time gaps between packets in that conversation | "do packets arrive evenly or in bursts?" |
| **HpHp** | the conversation between two sockets (IP + port) | "camera:5000 and server:80" |

**2. Five time windows (λ, "lambda").** How far back it looks:
λ = 5, 3, 1, 0.1, 0.01, from the last ~100 milliseconds to the last ~1 minute.
An attack looks different over a short window than over a long one.

**3. The statistics.** What is measured in each window:

| Statistic | In plain words | Can it be negative? |
|---|---|---|
| weight | about how many packets are in the window | no |
| mean | average packet size (in HH_jit: average time gap) | no |
| variance | how spread out those values are | no |
| radius | spread of both directions together | no |
| magnitude | size of both directions together | no |
| cov | do both directions go up and down together? | yes |
| pcc | same as cov, but always between -1 and 1 | yes, but only down to -1 |

MI, H and HH_jit look at **one** direction, so they only have the first 3 statistics.
HH and HpHp compare **two** directions (going and coming back), so they have all 7.

**Total:** 3 blocks × 5 windows × 3 stats = 45, plus 2 blocks × 5 windows × 7 stats = 70. **45 + 70 = 115.**

## Step 1. Get the smallest and largest value of every column

**What:** for each of the 115 columns, compute its minimum and maximum.

**Why:** the rules we will check later are about limits ("never below 0", "never above 1"). Min and max are all we need to check them.

**Why it is written this way:**
- We build all 230 calculations (115 min + 115 max) first and run them in **one** `.agg()`, so Spark reads the table **once**. One `.agg()` per column would read it 115 times.
- Bronze stores everything as text, so we convert to numbers with `try_cast`. If a value can't be converted it becomes empty (null) instead of crashing. For exploring, that is what we want.
- The result, `stats_row`, is a single row. You read it like a dictionary, for example `stats_row["_c36_min"]`.

In [0]:
from pyspark.sql import functions as F

FEATURES_TABLE = "kitsune_dev.bronze.syn_dos_features_raw"
N_FEATURES = 115

features_df = spark.table(FEATURES_TABLE)

# Bronze column names: _c0 .. _c114
feature_cols = [f"_c{i}" for i in range(N_FEATURES)]

# One min and one max expression per column (string -> double)
agg_exprs = []
for c in feature_cols:
    value = F.col(c).try_cast("double")
    agg_exprs.append(F.min(value).alias(f"{c}_min"))
    agg_exprs.append(F.max(value).alias(f"{c}_max"))

# Single pass over the table
stats_row = features_df.agg(*agg_exprs).first()


## Step 2. Quick test on two columns

**What:** look at the one column where the two layouts disagree most clearly.

**Why:** a **pcc** (correlation) is always between -1 and 1. Nothing else can make that true by force.
- If layout A is right, the pcc is in `_c36`.
- If layout B is right, the pcc is in `_c51`.

Whichever column stays between -1 and 1 tells us the winner.

**Result:** `_c36` stays between -0.13 and 0.14 (fits). `_c51` goes up to 5609, which is impossible for a pcc. **Layout B is ruled out.**

In [0]:
# Fingerprint: pcc must be within [-1, 1]
# Layout A (code order) -> pcc at _c36
# Layout B (paper order) -> pcc at _c51
for c in ["_c36", "_c51"]:
    print(c, "min =", stats_row[f"{c}_min"], "| max =", stats_row[f"{c}_max"])

_c36 min = -0.1313498805033678 | max = 0.13880258663999626
_c51 min = 1.0 | max = 5609.796711263494


## Step 3. Write down what each column should be (the "map")

**What:** build a list with 115 entries, one per column, saying what it should contain under layout A, as `(block, window, statistic)`.

**Result:** 115 entries. Entry 36 is `('HH', '5', 'pcc')` and entry 51 is `('HH', '0.1', 'weight')`, which matches what we saw in step 2.

In [0]:
# Building blocks of the Kitsune feature vector (layout A = code order)
LAMBDAS = ["5", "3", "1", "0.1", "0.01"]
STATS_1D = ["weight", "mean", "variance"]
STATS_2D = ["weight", "mean", "variance", "radius", "magnitude", "cov", "pcc"]

# Each block: (name, stats computed per lambda)
BLOCKS = [
    ("MI", STATS_1D),
    ("H", STATS_1D),
    ("HH", STATS_2D),
    ("HH_jit", STATS_1D),
    ("HpHp", STATS_2D),
]

# Expected content of each column, in order: (block, lambda, stat)
layout = []
for block, stats in BLOCKS:
    for lam in LAMBDAS:
        for stat in stats:
            layout.append((block, lam, stat))

print(len(layout))
print(layout[36])
print(layout[51])

115
('HH', '5', 'pcc')
('HH', '0.1', 'weight')


## Step 4. Rule 1: some statistics are never negative

**What:** for every column that the map says is a weight, mean, variance, radius or magnitude, check that its minimum is not below 0.

**Why:** you can't have a negative number of packets, a negative size or a negative spread. If one of these columns had a negative minimum, the map would be wrong.

**How to read the code:** it only prints when it finds something **impossible**. If everything is fine, it only prints `Done`.

**Result:** `Done`. All 95 columns pass.

In [0]:
# Stats that can never be negative
NON_NEGATIVE = ["weight", "mean", "variance", "radius", "magnitude"]

for i in range(N_FEATURES):
    block, lam, stat = layout[i]           # what this column should be
    col_min = stats_row[f"_c{i}_min"]      # smallest value in the data

    # Print only if the rule is broken
    if stat in NON_NEGATIVE and col_min < 0:
        print("PROBLEM:", f"_c{i}", block, lam, stat, col_min)

print("Done")

Done


## Step 5. Rule 2: pcc is always between -1 and 1

**What:** for the 10 columns that the map says are pcc, check that the minimum is not below -1 and the maximum is not above 1.

**Why:** that is how a correlation coefficient is defined. A pcc outside [-1, 1] would mean the map is wrong.

**Result:** `Done`. All 10 pcc columns pass.
(The 10 cov columns have no fixed limits, so there is nothing to check for them.)

In [0]:
# pcc must be within [-1, 1]
for i in range(N_FEATURES):
    block, lam, stat = layout[i]
    col_min = stats_row[f"_c{i}_min"]
    col_max = stats_row[f"_c{i}_max"]

    # Print only if the rule is broken
    if stat == "pcc" and (col_min < -1 or col_max > 1):
        print("PROBLEM:", f"_c{i}", block, lam, stat, col_min, col_max)

print("Done")

Done


## Step 6. Safety check: no NaN

**What:** check that no column has NaN ("Not a Number").

**Why:** any comparison with NaN gives `False`. So a column full of NaN would **pass steps 4 and 5 without being checked**. This step makes sure those "Done" results can be trusted.

In Spark, NaN counts as bigger than any number, so if a column had one, its max would be NaN. That's why we only need to look at the max.

**Result:** empty list. No NaN anywhere.

In [0]:
import math

# Columns whose max is NaN or None (comparisons above would silently miss them)
bad_cols = []
for i in range(N_FEATURES):
    col_max = stats_row[f"_c{i}_max"]
    if col_max is None or math.isnan(col_max):
        bad_cols.append(f"_c{i}")

print("Columns with NaN/None:", bad_cols)

Columns with NaN/None: []


## Results and conclusion

| Check | Columns | Result |
|---|---|---|
| Quick pcc test (`_c36` vs `_c51`) | 2 | layout B impossible |
| Never negative | 95 | all pass under layout A |
| pcc between -1 and 1 | 10 | all pass under layout A |
| No NaN | 115 | none found |

Two more clues that layout A is right:
- `_c51` (HH, window 0.1, weight) has a minimum of **exactly 1.0**. A window always contains at least the current packet.
- The MI and H mean columns have a minimum of **about 60 bytes**. That's the smallest Ethernet frame (64 bytes) without its 4-byte checksum, which captures usually leave out.

**Conclusion:** layout A (the order in the Kitsune code) is correct. The paper's Table II lists the blocks in a different order than the real data.
Silver will name the 115 columns using layout A.